Name: Dev Patel 

Course: DS4400 Data Mining and Machine Learning 1

Prof: Silvio Amir

University: Northeastern University

Problem 4: Cross Validation

1. Implement k-fold cross-validation from scratch.
2. Run CV for Logistic Regression and LDA with $k \in \{5, 10\}$.
3. Compare which model performs better.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

In [2]:
# Load SPAMBASE dataset
col_names = [
    'word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d', 'word_freq_our',
    'word_freq_over', 'word_freq_remove', 'word_freq_internet', 'word_freq_order', 'word_freq_mail',
    'word_freq_receive', 'word_freq_will', 'word_freq_people', 'word_freq_report', 'word_freq_addresses',
    'word_freq_free', 'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit',
    'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money', 'word_freq_hp',
    'word_freq_hpl', 'word_freq_george', 'word_freq_650', 'word_freq_lab', 'word_freq_labs',
    'word_freq_telnet', 'word_freq_857', 'word_freq_data', 'word_freq_415', 'word_freq_85',
    'word_freq_technology', 'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct',
    'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project', 'word_freq_re',
    'word_freq_edu', 'word_freq_table', 'word_freq_conference',
    'char_freq_;', 'char_freq_(', 'char_freq_[', 'char_freq_!', 'char_freq_$', 'char_freq_#',
    'capital_run_length_average', 'capital_run_length_longest', 'capital_run_length_total',
    'spam'
]

df = pd.read_csv('spambase.data', header=None, names=col_names)
X = df.drop('spam', axis=1).values
y = df['spam'].values

print(f"Dataset: X={X.shape}, y={y.shape}")

Dataset: X=(4601, 57), y=(4601,)


### Part 1: Implement k-Fold Cross-Validation

In [3]:
def k_fold_cv(model_class, X, y, k, **model_kwargs):
    """
    k-fold cross-validation.
    
    1. Shuffle and divide data into k partitions of equal size.
    2. For each fold i, train on k-1 partitions and test on partition i.
    3. Record validation error for each fold.
    4. Return average validation error.
    """
    n = len(y)
    indices = np.arange(n)
    np.random.seed(42)
    np.random.shuffle(indices)
    
    # Divide into k partitions
    fold_size = n // k
    folds = []
    for i in range(k):
        start = i * fold_size
        if i == k - 1:
            end = n  # last fold gets remaining samples
        else:
            end = start + fold_size
        folds.append(indices[start:end])
    
    errors = []
    for i in range(k):
        # Validation set is fold i
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        
        X_train_cv, y_train_cv = X[train_idx], y[train_idx]
        X_val_cv, y_val_cv = X[val_idx], y[val_idx]
        
        model = model_class(**model_kwargs)
        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict(X_val_cv)
        
        val_error = 1 - accuracy_score(y_val_cv, y_pred)
        errors.append(val_error)
        print(f"  Fold {i+1}/{k}: validation error = {val_error:.4f}")
    
    avg_error = np.mean(errors)
    print(f"  Average validation error: {avg_error:.4f}\n")
    return avg_error, errors

### Part 2: Run CV for Logistic Regression and LDA

In [ ]:
results = []

for k in [5, 10]:
    print(f"=== Logistic Regression, k={k} ===")
    avg_err_lr, _ = k_fold_cv(LogisticRegression, X, y, k, max_iter=10000, random_state=42)
    results.append({'Model': 'Logistic Regression', 'k': k, 'Avg Validation Error': round(avg_err_lr, 4)})
    
    print(f"=== LDA, k={k} ===")
    avg_err_lda, _ = k_fold_cv(LinearDiscriminantAnalysis, X, y, k)
    results.append({'Model': 'LDA', 'k': k, 'Avg Validation Error': round(avg_err_lda, 4)})

results_df = pd.DataFrame(results)
print("\nSummary:\n")
results_df

=== Logistic Regression, k=5 ===
  Fold 1/5: validation error = 0.0772
  Fold 2/5: validation error = 0.0707
  Fold 3/5: validation error = 0.0750
  Fold 4/5: validation error = 0.0685
  Fold 5/5: validation error = 0.0684
  Average validation error: 0.0719

=== LDA, k=5 ===
  Fold 1/5: validation error = 0.1185
  Fold 2/5: validation error = 0.1065
  Fold 3/5: validation error = 0.1196
  Fold 4/5: validation error = 0.1065
  Fold 5/5: validation error = 0.1118
  Average validation error: 0.1126

=== Logistic Regression, k=10 ===
  Fold 1/10: validation error = 0.0761
  Fold 2/10: validation error = 0.0739
  Fold 3/10: validation error = 0.0543


### Part 3: Comparison

**Observations:**

- **Logistic Regression vs LDA:** Both are linear classifiers, but they make different assumptions. LDA assumes features are normally distributed with equal covariance matrices across classes, while logistic regression makes no such assumption. The model with the lower average validation error performs better on this dataset.

- **Effect of k:** Using k=10 provides a less biased estimate of the validation error compared to k=5 because each fold uses more training data (90% vs 80%). However, k=10 has higher variance since the validation sets are smaller. The average errors for k=5 and k=10 should be similar if the models are stable.

- **Overall:** Cross-validation gives a more reliable estimate of model performance than a single train/test split, since it averages over multiple partitions of the data. The model with the lowest average CV error is preferred.